## OpenAI categorization

In [ ]:
import pandas as pd
comment_df = pd.read_csv(r'\data\comment.csv')
comment_df.head()

,media_id,assigned_niche,account_handle,timestamp,caption,comment_user,comment_text,comment_clean,positive_jerry,positive_nltk,comment_for_freebie
0,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,guanaha_tabeyes,"Because bright turquoise is a happy, it’s a Fl...",because bright turquoise is a happy it's a flo...,True,True,False
1,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,farrellclick,Savannah has gross muddy ocean water and we ar...,savannah has gross muddy ocean water and we ar...,False,False,False
2,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,huntfish70,White. Water is clear in the shallow end and f...,white water is clear in the shallow end and fa...,False,True,False
3,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,theboardman247,What is the pool color?,what is the pool color?,False,False,False
4,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,rajkamal_tarar,"Can""you""share @carolina__diaries",can you share,False,True,False


#### Remove non-substantive comments

In [ ]:
# Remove Emoji-only comments
import re
def is_question_marks_only(s: str) -> bool:
    """
    Returns True if the string consists only of groups of '?'
    separated by whitespace.
    """
    return bool(re.fullmatch(r"\?+(?:\s+\?+)*", s.strip()))

# Remove Comments with only "link" in the text
comment_df['comment_link'] = comment_df['comment_clean'].str.strip() == 'link'

In [ ]:
# Edit and save
non_freebie_comments = comment_df[~comment_df['comment_for_freebie'] & ~comment_df['comment_link'] & comment_df['comment_clean'].notna()]
non_freebie_comments['is_question_marks'] = non_freebie_comments['comment_clean'].apply(is_question_marks_only)
non_freebie_comments = non_freebie_comments[~non_freebie_comments['is_question_marks']]

non_freebie_comments.to_csv(r'\data\non_freebie_comments.csv',index=False)

/tmp/ipykernel_3223/3734865181.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_freebie_comments['is_question_marks'] = non_freebie_comments['comment_clean'].apply(is_question_marks_only)


#### Extract Comment Topics

In [ ]:
from openai import OpenAI
import pandas as pd
import json
from google.colab import userdata

key = userdata.get('OpenAI_API')
client = OpenAI(api_key = key)

def extract_keywords_batch(comments):
    numbered_comments = "\n".join(
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    )
    prompt = f"""
        There are exactly {len(batch)} Instagram comments.

        For each comment, return:
        - topics: up to 5 specific noun keywords. Exclude emotion words.
        - demands: creator-directed requests/questions, or null if none.
        - category: one of "Request", "Purchase Intent", "Confusion", "Pain Point", or "Other".
        - emotions: up to 5 emotion words, or ["Neutral"] if none.

        Return ONLY this JSON array:

        [
          {{"index":0,"topics":[...],"demands":null,"category":"Other","emotions":["Neutral"]}},
          {{"index":1,"topics":[...],"demands":["..."],"category":"Request","emotions":["curious"]}}
        ]

        COMMENTS:
        {numbered_comments}
        """
    response = client.chat.completions.create(
        model="gpt-4.1-mini-2025-04-14",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=3000,
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    print(raw)
    # Remove markdown fences if present
    if raw.startswith("```"):
        lines = raw.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        raw = "\n".join(lines)

    parsed = json.loads(raw)
    print(parsed)
    # Create one placeholder per input comment
    output = [
        {
            "topics": [],
            "demands": None,
            "category": "Other",
            "emotions": ["Neutral"]
        }
        for _ in comments
    ]

    for item in parsed:
        if not isinstance(item, dict):
            continue

        idx = item.get("index")

        if not isinstance(idx, int):
            continue

        if 0 <= idx < len(comments):
            output[idx] = {
                "topics": item.get("topics", []),
                "demands": item.get("demands", None),
                "category": item.get("category", "Other"),
                "emotions": item.get("emotions", ["Neutral"])
            }

    return output

# -----------------------------
# Apply to comment_df in batches
# -----------------------------

batch_size = 50
all_topics = []
all_demands = []
all_categories = []
all_emotions = []

comments = (
    non_freebie_comments["comment_clean"]
    .fillna("")
    .astype(str)
    .tolist()
)

for batch_start in range(0, len(comments), batch_size):
    print(batch_start)

    batch = comments[batch_start:batch_start + batch_size]

    try:
        batch_results = extract_keywords_batch(batch)

    except Exception as e:
        print(f"Failed batch starting at {batch_start}: {e}")

        batch_results = [
            {
                "topics": [],
                "demands": None,
                "category": "Other",
                "emotions": ["Neutral"]
            }
            for _ in batch
        ]

    all_topics.extend([x["topics"] for x in batch_results])
    all_demands.extend([x["demands"] for x in batch_results])
    all_categories.extend([x["category"] for x in batch_results])
    all_emotions.extend([x["emotions"] for x in batch_results])

non_freebie_comments["comment_topics"] = all_topics
non_freebie_comments["comment_demands"] = all_demands
non_freebie_comments["comment_category"] = all_categories
non_freebie_comments["comment_emotions"] = all_emotions

Streaming output truncated to the last 5000 lines.
16250
[
  {"index":0,"topics":["lawsuit"],"demands":["drop the lawsuit"],"category":"Request","emotions":["angry"]},
  {"index":1,"topics":["values","lawsuit"],"demands":["drop the lawsuit"],"category":"Request","emotions":["disappointed"]},
  {"index":2,"topics":["lawsuit"],"demands":["drop the lawsuit"],"category":"Request","emotions":["supportive"]},
  {"index":3,"topics":["lawsuit","Pattie Gonia","planet"],"demands":["drop the lawsuit"],"category":"Request","emotions":["concerned"]},
  {"index":4,"topics":["founder","lawsuit"],"demands":["how does the founder feel about the lawsuit?"],"category":"Request","emotions":["curious"]},
  {"index":5,"topics":["post","labor","climate activists","brand","founder","foundation","climate crisis"],"demands":null,"category":"Pain Point","emotions":["angry","disgusted"]},
  {"index":6,"topics":["boycott","Patagonia"],"demands":["boycott Patagonia"],"category":"Request","emotions":["angry"]},
  {"

In [ ]:
non_freebie_comments.head()

,media_id,assigned_niche,account_handle,timestamp,caption,comment_user,comment_text,comment_clean,positive_jerry,positive_nltk,comment_for_freebie,comment_link,is_question_marks,comment_topics,comment_demands,comment_category,comment_emotions
0,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,guanaha_tabeyes,"Because bright turquoise is a happy, it’s a Fl...",because bright turquoise is a happy it's a flo...,True,True,False,False,False,"[turquoise, color, Florida, state]",None,Other,"[happy, pride]"
1,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,farrellclick,Savannah has gross muddy ocean water and we ar...,savannah has gross muddy ocean water and we ar...,False,False,False,False,False,"[Savannah, muddy ocean water, pool, spectrum]",[curious where you would land on that spectrum],Request,[curious]
2,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,huntfish70,White. Water is clear in the shallow end and f...,white water is clear in the shallow end and fa...,False,True,False,False,False,"[white water, shallow end, blue, white sand be...",None,Other,[Neutral]
3,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,theboardman247,What is the pool color?,what is the pool color?,False,False,False,False,False,"[pool, color]",[what is the pool color?],Request,[curious]
4,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,rajkamal_tarar,"Can""you""share @carolina__diaries",can you share,False,True,False,False,False,[],[can you share],Request,[Neutral]


#### Extract Caption-level Topics

In [ ]:
captions_df = (
    non_freebie_comments[["media_id", "caption"]]
    .drop_duplicates()
)

In [ ]:
captions_df['id+caption'] = captions_df['media_id'].astype(str) + ' ' + captions_df['caption']

In [ ]:
def extract_keywords_batch(comments):
    numbered_comments = "\n".join(
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    )
    prompt = f"""
    There are exactly {len(comments)} captions.

    For each captions, extract 3-8 keywords.

    Return ONLY a JSON array of objects:

    [
      {{"index": 0, "keywords": [...]}},
      {{"index": 1, "keywords": [...]}}
    ]

    COMMENTS:
    {numbered_comments}
    """
    response = client.chat.completions.create(
        model="gpt-4.1-mini-2025-04-14",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=3000,
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    # Remove markdown fences if present
    if raw.startswith("```"):
        lines = raw.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        raw = "\n".join(lines)

    parsed = json.loads(raw)
    # Create one slot per input comment
    output = [[] for _ in comments]
    for item in parsed:
        if not isinstance(item, dict):
            continue
        idx = item.get("index")
        if not isinstance(idx, int):
            continue
        if 0 <= idx < len(comments):
            output[idx] = item.get("keywords", [])
    return output

In [ ]:
batch_size = 50
all_keywords = []

captions = (
    captions_df["id+caption"]
    .tolist()
)

for batch_start in range(0, len(captions), batch_size):
    print(batch_start)
    batch = captions[batch_start:batch_start + batch_size]
    try:
        batch_keywords = extract_keywords_batch(batch)
    except Exception as e:
        print(
            f"Failed batch starting at {batch_start}: {e}"
        )
        batch_keywords = [[] for _ in batch]

    all_keywords.extend(batch_keywords)

In [ ]:
captions_df['caption_keywords'] = all_keywords
caption_keyword_dict = captions_df.set_index('media_id')['caption_keywords'].to_dict()

In [ ]:
non_freebie_comments['caption_keywords'] = non_freebie_comments['media_id'].map(caption_keyword_dict)
non_freebie_comments.to_csv(r'\data\comments_openai.csv',index=False)

#### Results

In [ ]:
import pandas as pd
non_freebie_comments = pd.read_csv(r'\data\comments_openai.csv')
non_freebie_comments.head(20)

,media_id,assigned_niche,account_handle,timestamp,caption,comment_user,comment_text,comment_clean,positive_jerry,positive_nltk,comment_for_freebie,comment_link,is_question_marks,comment_topics,comment_demands,comment_category,comment_emotions,caption_keywords,comment_keywords
0,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,guanaha_tabeyes,"Because bright turquoise is a happy, it’s a Fl...",because bright turquoise is a happy it's a flo...,True,True,False,False,False,"['turquoise', 'color', 'Florida', 'state']",NaN,Other,"['happy', 'pride']","['turquoise pools', 'Malibu', 'hot take']","['bright turquoise', 'happy', 'Florida', 'beau..."
1,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,farrellclick,Savannah has gross muddy ocean water and we ar...,savannah has gross muddy ocean water and we ar...,False,False,False,False,False,"['Savannah', 'muddy ocean water', 'pool', 'spe...",['curious where you would land on that spectrum'],Request,['curious'],"['turquoise pools', 'Malibu', 'hot take']","['Savannah', 'muddy ocean water', 'pool', 'spe..."
2,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,huntfish70,White. Water is clear in the shallow end and f...,white water is clear in the shallow end and fa...,False,True,False,False,False,"['white water', 'shallow end', 'blue', 'white ...",NaN,Other,['Neutral'],"['turquoise pools', 'Malibu', 'hot take']","['white water', 'clear', 'shallow end', 'cryst..."
3,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,theboardman247,What is the pool color?,what is the pool color?,False,False,False,False,False,"['pool', 'color']",['what is the pool color?'],Request,['curious'],"['turquoise pools', 'Malibu', 'hot take']","['pool color', 'question']"
4,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,rajkamal_tarar,"Can""you""share @carolina__diaries",can you share,False,True,False,False,False,[],['can you share'],Request,['Neutral'],"['turquoise pools', 'Malibu', 'hot take']","['share', 'information', 'request']"
5,2,Home Design,chrislovesjulia,2026-05-21T21:30:10+0000,Why are there so many turquoise pools?,instafrances,I agree,i agree,False,True,False,False,False,"['Mississippi', 'pool', 'color', 'snakes']",['i want a light colored pool so i can see if ...,Request,['concerned'],"['turquoise pools', 'many', 'question']","['location', 'Mississippi', 'light colored poo..."
6,2,Home Design,chrislovesjulia,2026-05-21T21:30:10+0000,Why are there so many turquoise pools?,kenna_vowell,We live in MS and I want a light colored pool ...,we live in ms and i want a light colored pool ...,False,True,False,False,False,"['moody pool', 'light pool', 'child', 'drownin...",NaN,Other,"['sad', 'grateful']","['turquoise pools', 'many', 'question']","['moody pool', 'light pool', 'child safety', '..."
7,2,Home Design,chrislovesjulia,2026-05-21T21:30:10+0000,Why are there so many turquoise pools?,caitlinpeterson_,I love a moody pool but we chose light because...,i love a moody pool but we chose light because...,True,True,False,False,False,"['people', 'DC', 'notes']",NaN,Other,['Neutral'],"['turquoise pools', 'many', 'question']","['people', 'Washington DC', 'take notes', 'adv..."
8,2,Home Design,chrislovesjulia,2026-05-21T21:30:10+0000,Why are there so many turquoise pools?,2olga,People in DC should take some notes,people in dc should take some notes,False,False,False,False,False,['pool'],NaN,Other,['admiration'],"['turquoise pools', 'many', 'question']","['gorgeous pool', 'compliment']"
9,2,Home Design,chrislovesjulia,2026-05-21T21:30:10+0000,Why are there so many turquoise pools?,polly.drew,Your pool is gorgeous. Period.,your pool is gorgeous period,False,True,False,False,False,"['pool color', 'trend', 'pressure']",NaN,Other,['supportive'],"['turqu

In [ ]:
media_info = pd.read_csv('media_info.csv')

In [ ]:
media_info.columns

Index(['Unnamed: 0', 'assigned_niche', 'account_handle', 'timestamp',
       'media_type', 'media_product_type', 'like_count', 'views'],
      dtype='object')

In [ ]:
media_info.rename(columns={'Unnamed: 0': 'media_id'}, inplace=True)

In [ ]:
media_info = media_info.drop(['assigned_niche', 'account_handle', 'timestamp'],axis=1)

In [ ]:
len(non_freebie_comments)

20615

In [ ]:
full = non_freebie_comments.merge(media_info, on='media_id')
len(full)

20615

In [ ]:
full.head()

,media_id,assigned_niche,account_handle,timestamp,caption,comment_user,comment_text,comment_clean,positive_jerry,positive_nltk,...,comment_link,is_question_marks,comment_topics,comment_demands,comment_category,comment_emotions,media_type,media_product_type,like_count,views
0,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,guanaha_tabeyes,"Because bright turquoise is a happy, it’s a Fl...",because bright turquoise is a happy it's a flo...,True,True,...,False,False,"['turquoise', 'color', 'Florida', 'state']",NaN,Other,"['happy', 'pride']",VIDEO,REELS,790.0,37850.0
1,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,farrellclick,Savannah has gross muddy ocean water and we ar...,savannah has gross muddy ocean water and we ar...,False,False,...,False,False,"['Savannah', 'muddy ocean water', 'pool', 'spe...",['curious where you would land on that spectrum'],Request,['curious'],VIDEO,REELS,790.0,37850.0
2,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,huntfish70,White. Water is clear in the shallow end and f...,white water is clear in the shallow end and fa...,False,True,...,False,False,"['white water', 'shallow end', 'blue', 'white ...",NaN,Other,['Neutral'],VIDEO,REELS,790.0,37850.0
3,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,theboardman247,What is the pool color?,what is the pool color?,False,False,...,False,False,"['pool', 'color']",['what is the pool color?'],Request,['curious'],VIDEO,REELS,790.0,37850.0
4,0,Home Design,chrislovesjulia,2026-05-26T18:39:17+0000,My hot take: turquoise pools should be saved f...,rajkamal_tarar,"Can""you""share @carolina__diaries",can you share,False,True,...,False,False,[],['can you share'],Request,['Neutral'],VIDEO,REELS,790.0,37850.0


In [ ]:
from collections import Counter, defaultdict
import pandas as pd

def get_top_keywords(df, keyword_col, num_keywords=10, agg_col=None, agg_method='sum'):
    """
    Extracts and returns the top N most frequent keywords from the 'keyword_col' column
    of a DataFrame, either by count or by an aggregation (sum, mean, median, count distinct)
    of a specified aggregation column.

    Args:
        df (pd.DataFrame): The input DataFrame.
        keyword_col (str): The name of the column containing lists of keywords.
        num_keywords (int): The number of top keywords to return. If None, all available values are shown.
        agg_col (str, optional): The name of a numerical column to use for aggregation.
                                  If None, keywords are ranked by count.
                                  If specified, keywords are ranked by the aggregation
                                  of values in this column for each keyword.
        agg_method (str): The aggregation method to use when 'agg_col' is specified.
                          Can be 'sum', 'mean', 'median', 'count', or 'count distinct'.
                          Defaults to 'sum'. If 'agg_col' is None, this is forced to 'count'.

    Returns:
        pd.DataFrame: A DataFrame with 'keyword' and a dynamically named value column
                      (e.g., 'count', 'sum', 'mean', 'median', 'distinct_count'),
                      representing the top N keywords and their aggregated values.
    """
    if keyword_col not in df.columns or df[keyword_col].empty:
        print(f"DataFrame does not contain '{keyword_col}' column or it is empty.")
        return pd.DataFrame(columns=['keyword', 'value'])

    # Validate agg_col if provided
    if agg_col is not None:
        if agg_col not in df.columns:
            print(f"Aggregation column '{agg_col}' not found in DataFrame.")
            return pd.DataFrame(columns=['keyword', 'value'])
        if not pd.api.types.is_numeric_dtype(df[agg_col]):
            print(f"Aggregation column '{agg_col}' is not numeric.")
            return pd.DataFrame(columns=['keyword', 'value'])

    # Handle agg_method based on agg_col presence
    if agg_col is None:
        if agg_method != 'count':
            print(f"Warning: 'agg_col' is None, 'agg_method' is forced to 'count'.")
        agg_method_actual = 'count'
    else:
        valid_agg_methods = ['sum', 'mean', 'median', 'count', 'count distinct']
        if agg_method not in valid_agg_methods:
            print(f"Warning: Invalid 'agg_method' '{agg_method}' for 'agg_col'. Defaulting to 'sum'.")
            agg_method_actual = 'sum'
        else:
            agg_method_actual = agg_method

    keyword_data_points = [] # Stores (keyword, value_from_agg_col_or_1) for each instance

    for index, row in df.iterrows():
        keyword_list_raw = row[keyword_col]

        current_value_for_instance = 1 # Default for counting or when agg_col is None
        if agg_col is not None and agg_method_actual != 'count':
            current_value_for_instance = row[agg_col]

        if pd.isna(keyword_list_raw):
            continue

        actual_keyword_list = []
        if isinstance(keyword_list_raw, list):
            actual_keyword_list = keyword_list_raw
        elif isinstance(keyword_list_raw, str):
            try:
                parsed_list = eval(keyword_list_raw)
                if isinstance(parsed_list, list):
                    actual_keyword_list = parsed_list
            except (SyntaxError, NameError):
                pass # Skip if parsing fails

        if not pd.isna(current_value_for_instance):
            for keyword in actual_keyword_list:
                keyword_data_points.append((keyword, current_value_for_instance))

    if not keyword_data_points:
        print(f"No valid keyword data found in '{keyword_col}' column for aggregation.")
        return pd.DataFrame(columns=['keyword', 'value'])

    # Perform aggregation based on agg_method_actual
    grouped_data = defaultdict(list)
    for keyword, value in keyword_data_points:
        grouped_data[keyword].append(value)

    aggregated_results = {}
    for keyword, values_list in grouped_data.items():
        if not values_list: # Handle cases where a keyword might appear but have no associated non-NaN values
            aggregated_results[keyword] = 0 # Or some other appropriate default
            continue

        if agg_method_actual == 'sum':
            aggregated_results[keyword] = sum(values_list)
        elif agg_method_actual == 'mean':
            aggregated_results[keyword] = sum(values_list) / len(values_list)
        elif agg_method_actual == 'median':
            aggregated_results[keyword] = pd.Series(values_list).median()
        elif agg_method_actual == 'count':
            aggregated_results[keyword] = len(values_list)
        elif agg_method_actual == 'count distinct':
            aggregated_results[keyword] = len(set(values_list))

    # Sort and create DataFrame
    sorted_keywords = sorted(
        aggregated_results.items(),
        key=lambda item: item[1],
        reverse=True
    )

    # Apply num_keywords logic: if None, show all; otherwise, show top N
    if num_keywords is None:
        top_items = sorted_keywords
    else:
        top_items = sorted_keywords[:num_keywords]

    # Define column name for the aggregated value
    value_col_name = agg_method_actual if agg_method_actual != 'count distinct' else 'distinct_count'
    results_df = pd.DataFrame(top_items, columns=['keyword', value_col_name])

    return results_df

Likes

In [ ]:
get_top_keywords(full, keyword_col='caption_keywords', agg_col='like_count', agg_method='sum')

,keyword,sum
0,POV,90203309.0
1,overstimulated,90203309.0
2,long way,90203309.0
3,parenting,37919071.0
4,mother's day,34767082.0
5,self-doubt,34433712.0
6,mom hacks,34431750.0
7,mom life,34431732.0
8,kids love,34431732.0
9,contribution,30444533.0


In [ ]:
# Look at average
get_top_keywords(full, keyword_col='caption_keywords', agg_col='like_count', agg_method='mean')

,keyword,mean
0,POV,230699.000000
1,overstimulated,230699.000000
2,long way,230699.000000
3,mom life,168783.000000
4,kids love,168783.000000
5,mom hacks,168231.480392
6,self-doubt,160380.009346
7,mother's day,149422.089337
8,Miami casting,103272.000000
9,photobooth,103272.000000


Views

In [ ]:
get_top_keywords(full, keyword_col='caption_keywords', agg_col='views', agg_method='sum')

,keyword,sum
0,window film,4.754707e+09
1,5/19,4.754707e+09
2,renter-friendly,4.754707e+09
3,launch,4.754173e+09
4,2 patterns,4.754173e+09
5,under $30,4.754173e+09
6,Miami casting,1.708124e+09
7,photobooth,1.708124e+09
8,Smooth,1.096052e+09
9,chaos2themax2.0,1.096052e+09


In [ ]:
get_top_keywords(full, keyword_col='caption_keywords', agg_col='views', agg_method='mean')

,keyword,mean
0,Miami casting,4.679792e+06
1,photobooth,4.679792e+06
2,launch,4.084341e+06
3,2 patterns,4.084341e+06
4,under $30,4.084341e+06
5,window film,4.074299e+06
6,5/19,4.074299e+06
7,renter-friendly,4.074299e+06
8,Smooth,3.859337e+06
9,chaos2themax2.0,3.859337e+06


Comment Counts

In [ ]:
get_top_keywords(full, keyword_col='caption_keywords', agg_method='count')

,keyword,count
0,photography,730
1,Guizhou,601
2,TheContainerStore,576
3,natgeo,540
4,Canon USA,527
5,parenting,526
6,contribution,525
7,2016 vibes,518
8,2026,518
9,thenorthface,501


In [ ]:
get_top_keywords(full, keyword_col='comment_topics', agg_method='count')

,keyword,count
0,lawsuit,827
1,person,533
2,comment,438
3,birthday,344
4,appearance,343
5,link,312
6,work,309
7,photo,300
8,content,298
9,photos,280


In [ ]:
get_top_keywords(full, keyword_col='comment_demands', agg_method='count')

,keyword,count
0,drop the lawsuit,720
1,link please,130
2,list,42
3,links please,20
4,link,19
5,boycott Patagonia,12
6,send me this post,9
7,links,8
8,do better,6
9,drop the suit,6


In [ ]:
full.comment_category.value_counts()

,count
comment_category,
Other,16040
Request,3442
Pain Point,838
Confusion,185
Purchase Intent,110
